# Periocular Gender and Age Prediction

This notebook is now a lightweight companion to the CLI workflow rather than the primary source of truth.

Use the scripts in `scripts/` for the real pipeline. Use this notebook to:

- review the refreshed workflow
- inspect saved run results
- load the best checkpoints for quick interactive experiments


## Recommended Order

1. Prepare datasets with the CLI scripts.
2. Train models with `run_age_baselines.sh` or `run_gender_baselines.sh`.
3. Compare runs with `scripts/compare_runs.py`.
4. Use this notebook afterward for inspection and demo inference.


In [ ]:
from pathlib import Path
import json
import sys

import torch
from PIL import Image
from torchvision import transforms

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.perigender import PeriGender, PeriGenderV2
from models.periage import PeriAge, PeriAgeV2, PeriAgeResNet34

print('repo_root =', REPO_ROOT)
print('cuda_available =', torch.cuda.is_available())


## CLI Commands

Run these in a shell from the repository root.


In [ ]:
cli_commands = {
    'prepare_gender': "python scripts/prepare_ubipr_gender.py --raw-dir data/raw/ubipr/UBIPeriocular --output-dir data/processed/ubipr_gender --test-size 0.10 --seed 42 --balance-train --split-by-subject",
    'extract_age_crops': "python scripts/extract_utkface_periocular.py --raw-dir data/raw/utkface --output-dir data/processed/utkface_periocular_raw --strict",
    'prepare_age': "python scripts/prepare_utkface_age.py --raw-dir data/processed/utkface_periocular_raw --output-dir data/processed/utkface_age --test-size 0.20 --seed 42",
    'compare_gender': "python scripts/compare_runs.py --runs-dir runs/gender/ubipr",
    'compare_age': "python scripts/compare_runs.py --runs-dir runs/age/periocular",
    'age_sweep': "bash run_age_baselines.sh",
    'gender_sweep': "bash run_gender_baselines.sh",
}

cli_commands


## Canonical Runs Used In The Repo Docs


In [ ]:
canonical_runs = {
    'gender_baseline': REPO_ROOT / 'runs/gender/ubipr/baselines/resnet18_e30/20260330_212202/metrics.json',
    'gender_custom': REPO_ROOT / 'runs/gender/ubipr/custom/perigender_v2_adamw/20260331_004845/metrics.json',
    'age_baseline': REPO_ROOT / 'runs/age/periocular/baselines/resnet34/20260330_151158/metrics.json',
    'age_custom_v2': REPO_ROOT / 'runs/age/periocular/custom/periage_v2_bs32/20260330_174946/metrics.json',
    'age_best': REPO_ROOT / 'runs/age/periocular/hybrid/periage_resnet34_ft/20260330_202332/metrics.json',
}

for name, path in canonical_runs.items():
    print(name, 'exists =' , path.exists(), path)


In [ ]:
def load_best(metrics_path: Path):
    payload = json.loads(metrics_path.read_text())
    best = max(payload['history'], key=lambda row: row['test_acc'])
    return {
        'model': payload['model'],
        'lr': payload.get('lr'),
        'best_epoch': best['epoch'],
        'best_test_acc': best['test_acc'],
        'final_test_acc': payload['history'][-1]['test_acc'],
    }

summary = {name: load_best(path) for name, path in canonical_runs.items() if path.exists()}
summary


## Load A Saved Checkpoint For Interactive Inference

Update the `checkpoint_path` and `image_path` values below before running.


In [ ]:
checkpoint_path = REPO_ROOT / 'runs/gender/ubipr/custom/perigender_v2_adamw/20260331_004845/best.pt'
image_path = REPO_ROOT / 'example.jpg'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

payload = torch.load(checkpoint_path, map_location='cpu')
model_name = payload['model_name']

if model_name == 'perigender':
    model = PeriGender()
elif model_name == 'perigender_v2':
    model = PeriGenderV2()
elif model_name == 'periage':
    model = PeriAge()
elif model_name == 'periage_v2':
    model = PeriAgeV2()
elif model_name == 'periage_resnet34':
    model = PeriAgeResNet34(weights=None)
else:
    raise ValueError(f'Unsupported model: {model_name}')

model.load_state_dict(payload['state_dict'])
model = model.to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

if image_path.exists():
    image = Image.open(image_path).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        pred = int(torch.argmax(logits, dim=1).item())
    print('model =', model_name)
    print('predicted_class_index =', pred)
else:
    print('Update image_path before running inference:', image_path)


## Notes

- The CLI scripts remain the authoritative workflow.
- The age pipeline now uses strict periocular crops, not full-face UTKFace images.
- The gender pipeline now uses subject-level splitting on UBIPr.
